# FLAME Mesh Repair & Reconstruction Service

This notebook provides a GPU-accelerated mesh repair pipeline using FLAME 3D face model.

**Modes:**
- `repair_only`: Basic mesh repair (fix normals, fill holes, remove degenerate faces)
- `flame_fit`: Fit FLAME model to input mesh, output clean FLAME topology
- `flame_blend`: Blend original mesh with FLAME fitting result

In [ ]:
!pip install -q torch numpy trimesh scipy open3d tqdm chumpy pyrender

In [ ]:
!git clone https://github.com/kk20250706/demo2026.git 2>/dev/null || true
import sys
sys.path.insert(0, '/content/demo2026')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Upload Files

Upload your FBX mesh file and (optionally) the FLAME model file (`generic_model.pkl`).

Download FLAME model from: https://flame.is.tue.mpg.de/download.php

In [ ]:
from google.colab import files
import os

os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Upload your mesh file (FBX/OBJ/PLY/STL):')
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
input_path = f'input/{input_filename}'
with open(input_path, 'wb') as f:
    f.write(uploaded[input_filename])
print(f'Saved to {input_path}')

In [ ]:
print("Upload generic_model.pkl (required for flame_blend mode):")
uploaded_model = files.upload()
model_name = list(uploaded_model.keys())[0]
with open(f"models/{model_name}", "wb") as f:
    f.write(uploaded_model[model_name])
print(f"FLAME model saved to models/{model_name}")


## 2. Run Mesh Repair

In [ ]:
from flame_repair.pipeline import MeshRepairPipeline

#@markdown ### Configuration
MODE = 'flame_blend'  #@param ['repair_only', 'flame_fit', 'flame_blend']
SMOOTH_ITERS = 2  #@param {type: 'integer'}
FLAME_ITERS = 500  #@param {type: 'integer'}
FLAME_MODEL_PATH = 'models/generic_model.pkl'  #@param {type: 'string'}

output_path = f'output/repaired_{input_filename}'

flame_path = FLAME_MODEL_PATH if MODE != 'repair_only' else None
pipeline = MeshRepairPipeline(flame_model_path=flame_path, device='auto')

result_mesh = pipeline.run(
    input_path=input_path,
    output_path=output_path,
    mode=MODE,
    smooth_iters=SMOOTH_ITERS,
    flame_iters=FLAME_ITERS,
)

## 3. Visualize Result

In [ ]:
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def plot_mesh(mesh, title='Mesh', max_faces=5000):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    verts = mesh.vertices
    faces = mesh.faces
    if len(faces) > max_faces:
        idx = np.random.choice(len(faces), max_faces, replace=False)
        faces = faces[idx]
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2],
                    triangles=faces, alpha=0.6, edgecolor='gray', linewidth=0.1)
    ax.set_title(f'{title} ({len(mesh.vertices)} verts, {len(mesh.faces)} faces)')
    plt.tight_layout()
    plt.show()

plot_mesh(result_mesh, 'Repaired Mesh')

## 4. Download Result

In [ ]:
import glob

output_files = glob.glob('output/*')
for f in output_files:
    print(f'Downloading: {f}')
    files.download(f)